# AskSage API connectivity test

This notebook loads the project `.env` file without displaying secret values, authenticates with the approved AskSage instance, lists account-available models and their API-reported modalities, and sends one minimal text-only prompt. It uses the project AskSage client so host validation, timeouts, retries, and error handling remain consistent with the application.

Run the cells from top to bottom. A successful run makes two API requests after authentication: one to list models and one text completion.

In [ ]:
#pip install truststore

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """Locate the repository root without hard-coding a local path."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Could not locate the project root containing pyproject.toml.")


def load_project_dotenv(dotenv_path: Path) -> None:
    """Load simple KEY=VALUE entries without logging their values."""
    if not dotenv_path.is_file():
        raise FileNotFoundError(f"Expected environment file was not found: {dotenv_path}")

    for raw_line in dotenv_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.removeprefix("export ").strip()
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in {"'", '"'}:
            value = value[1:-1]
        if key:
            os.environ.setdefault(key, value)


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
load_project_dotenv(PROJECT_ROOT / ".env")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from cepe_fynsp.asksage.client import (
    AskSageClient,
    AskSageConfigurationError,
    AskSageError,
)

required = ("ASKSAGE_INSTANCE",)
missing = [name for name in required if not os.getenv(name)]
if not os.getenv("ASKSAGE_ACCESS_TOKEN") and not (
    os.getenv("ASKSAGE_EMAIL") and os.getenv("ASKSAGE_API_KEY")
):
    missing.extend(("ASKSAGE_ACCESS_TOKEN or ASKSAGE_EMAIL + ASKSAGE_API_KEY",))
if missing:
    raise AskSageConfigurationError(
        "Missing AskSage configuration in .env: " + ", ".join(missing)
    )

client = AskSageClient.from_env()
print(f"Prepared a client for the approved host: {client.config.api_host}")


Prepared a client for the approved host: api.nnsa.asksage.ai


In [ ]:
from collections.abc import Iterable, Mapping
from typing import Any


def flatten_modalities(value: Any) -> list[str]:
    """Turn API capability fields into readable modality labels."""
    if isinstance(value, str):
        return [value]
    if isinstance(value, Mapping):
        return [str(name) for name, enabled in value.items() if enabled]
    if isinstance(value, Iterable):
        return [str(item) for item in value]
    return []


def reported_modalities(model: Mapping[str, Any]) -> str:
    """Read modality/capability fields only when AskSage includes them."""
    modality_fields = (
        "modalities",
        "modality",
        "input_modalities",
        "output_modalities",
        "supported_modalities",
        "capabilities",
        "features",
    )
    labels: list[str] = []
    for field in modality_fields:
        labels.extend(flatten_modalities(model.get(field)))
    return ", ".join(dict.fromkeys(labels)) if labels else "not reported by AskSage"


def extract_model_rows(payload: Mapping[str, Any]) -> list[dict[str, str]]:
    """Normalize either documented AskSage model-list response shape."""
    candidates = payload.get("data") or payload.get("response") or payload.get("models") or []
    if isinstance(candidates, Mapping):
        candidates = [candidates]
    if not isinstance(candidates, list):
        raise AskSageConfigurationError("AskSage returned an unrecognized model-list shape.")

    rows: list[dict[str, str]] = []
    for item in candidates:
        if isinstance(item, str):
            rows.append({"model": item, "modalities": "not reported by AskSage"})
            continue
        if isinstance(item, Mapping):
            model_id = item.get("id") or item.get("model") or item.get("name") or "unnamed model"
            rows.append({"model": str(model_id), "modalities": reported_modalities(item)})
    return rows


def print_model_table(rows: list[dict[str, str]]) -> None:
    if not rows:
        print("AskSage authenticated successfully, but no models were returned for this account.")
        return
    model_width = max(len("Model"), *(len(row["model"]) for row in rows))
    modality_width = max(len("Modalities"), *(len(row["modalities"]) for row in rows))
    print(f"{'Model':<{model_width}}  {'Modalities':<{modality_width}}")
    print(f"{'-' * model_width}  {'-' * modality_width}")
    for row in rows:
        print(f"{row['model']:<{model_width}}  {row['modalities']:<{modality_width}}")


try:
    # The API model-list response sometimes contains names only. This notebook does
    # not infer image/audio capability from a model name when metadata is absent.
    access_token = client.get_access_token()
    models_payload = client._post_json(
        f"{client.config.server_base_url}/get-models",
        headers={"x-access-tokens": access_token, "Content-Type": "application/json"},
        body={},
    )
    model_rows = extract_model_rows(models_payload)
except AskSageError as exc:
    print(f"AskSage connection test failed: {type(exc).__name__}: {exc}")
    raise

print("AskSage connection established and model-list request completed.")
print_model_table(model_rows)
print("\nNote: AskSage's model-list API does not always return modalities.")
print("'not reported by AskSage' means capability was absent from this account response.")


In [ ]:
PROMPT = "Reply with exactly: AskSage API connectivity test passed."

try:
    completion = client.chat_completion(
        [{"role": "user", "content": PROMPT}],
        temperature=0,
        max_tokens=32,
    )
    message = completion["choices"][0]["message"]["content"]
except (AskSageError, IndexError, KeyError, TypeError) as exc:
    print(f"AskSage text-prompt test failed: {type(exc).__name__}: {exc}")
    raise

print("AskSage text response:")
print(message)


In [ ]:
payload = client._post_json(
    f"{client.config.user_base_url}/get-token-with-api-key",
    headers={"Content-Type": "application/json"},
    body={"email": client.config.email, "api_key": client.config.api_key},
)

print("Top-level response keys:", sorted(payload))
print("Value types:", {key: type(value).__name__ for key, value in payload.items()})

In [ ]:
nested = payload["response"]

print("AskSage status:", payload["status"])
print("Nested response keys:", sorted(nested))
print("Nested value types:", {key: type(value).__name__ for key, value in nested.items()})

In [ ]:
nested = payload.get("response", {})
token = (
    payload.get("access_token")
    or payload.get("token")
    or (nested.get("access_token") if isinstance(nested, dict) else None)
    or (nested.get("token") if isinstance(nested, dict) else None)
)